# nada

> Wav2vec 2.0 speech recognition on Modular MAX.


#| export
[Wav2vec 2.0](https://arxiv.org/abs/2006.11477) on [Modular MAX](https://docs.modular.com/max/). This is the base model fine-tuned on 960 hours of LibriSpeech for character-level CTC: a convolutional feature encoder, a convolutional position embedding, and a 12-layer transformer, with a linear head that scores 32 characters for every 20 ms of audio. The weights are the Hugging Face release `facebook/wav2vec2-base-960h`.

The layout follows `solid_lens`. `config` is the architecture, `weights` binds a checkpoint, and `compile_model` traces the module for one batch size and one clip length. The graph is a MAX module (`max.experimental.nn`): the same `forward` runs eagerly or compiled. Module and parameter names match the Hugging Face implementation, so its state dict loads without renaming.


## Install

```sh
pixi install
pixi run convert
```

`convert` downloads the checkpoint and writes `models/wav2vec2.safetensors` plus the character vocabulary. The Hugging Face file is already safetensors, so conversion does not need PyTorch. It folds the position convolution's weight norm into one weight.


## Transcribe

```sh
pixi run transcribe clip.wav
```

The clip must be 16 kHz, 16-bit PCM WAV. The same thing from Python:

```python
from nada.features import read_wav
from nada.model import build, compile_model, load_checkpoint, transcribe

clip = read_wav("clip.wav")
model = build()
load_checkpoint(model)
compiled = compile_model(model, batch=1, num_samples=clip.size)
print(transcribe(model, compiled, [clip])[0])
```

`compiled` takes a float32 waveform of shape `(batch, num_samples)` and returns CTC logits `(batch, frames, 32)` and the final hidden states `(batch, frames, 768)`. The graph is compiled for one clip length, so recompile when it changes. `prepare` normalizes each clip to zero mean and unit variance. Clips in a batch are zero-padded to the longest one; the base checkpoint takes no attention mask, so batch clips of similar length.


## Check

```sh
pixi run smoke
pixi run -e reference parity
```

`smoke` compiles a two-layer model and checks that the logits are finite and the right shape. `parity` compares that same model, weight for weight, with Hugging Face, and compares the real checkpoint too once it is converted.
